# 🗂️ Week 6: Text Clustering

## Tujuan Pembelajaran:
1. Memahami konsep **Text Clustering** — mengelompokkan dokumen teks berdasarkan kemiripan kontennya
2. Menerapkan tiga metode vektorisasi teks: **TF-IDF**, **Bag of Words (BoW)**, dan **Cosine Similarity**
3. Menggunakan algoritma **KMeans** untuk mengelompokkan dokumen
4. Menentukan jumlah klaster optimal dengan **Elbow/WCSS** dan **Silhouette Method**
5. Memvisualisasikan klaster menggunakan **PCA** (Principal Component Analysis)

---
> 📂 **Dataset**: Honest Review (`cleandata.csv`) — kolom `text_final` (teks yang sudah dipreproses)
> File ini berada di folder `Week 3/` dalam workspace yang sama.

## 1) Install Library

Install semua library yang dibutuhkan untuk clustering dan visualisasi.

In [ ]:
!pip install scikit-learn pandas numpy matplotlib

## 2) Import Library

Import semua modul yang diperlukan:
- `TfidfVectorizer` & `CountVectorizer` untuk representasi teks
- `KMeans` untuk algoritma clustering
- `silhouette_score`, `calinski_harabasz_score`, `davies_bouldin_score` untuk evaluasi
- `PCA` untuk reduksi dimensi
- `cosine_similarity` untuk menghitung kemiripan antar dokumen

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

print("Library berhasil diimport!")

## 3) Load Dataset

Load dataset Honest Review dari `cleandata.csv`. Kolom `text_final` berisi teks ulasan yang sudah melalui proses preprocessing (tokenisasi, stopword removal, stemming).

In [ ]:
file_path = '../Week 3/cleandata.csv'
df = pd.read_csv(file_path)

print(f"Jumlah data: {len(df)}")
print(f"Kolom: {list(df.columns)}")
df['text_final'].head()

## 4) Fungsi Pembantu (Helper Functions)

Definisikan fungsi-fungsi yang akan digunakan berulang kali:
- `perform_clustering()` — melatih KMeans dan mengembalikan metrik evaluasi
- `get_top_terms()` — menampilkan kata kunci teratas dari setiap klaster
- `count_documents_per_cluster()` — menghitung jumlah dokumen per klaster

In [ ]:
def perform_clustering(X, num_clusters):
    """
    Melakukan KMeans clustering dan mengembalikan model serta metrik evaluasi.
    
    Parameters:
        X           : matriks fitur (sparse atau dense)
        num_clusters: jumlah klaster (k)
    Returns:
        (kmeans_model, labels, silhouette, ch_score, db_score)
    """
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    sil   = silhouette_score(X, labels)
    ch    = calinski_harabasz_score(X.toarray() if hasattr(X, 'toarray') else X, labels)
    db    = davies_bouldin_score(X.toarray() if hasattr(X, 'toarray') else X, labels)
    return kmeans, labels, sil, ch, db


def get_top_terms(vectorizer, kmeans_model, n=10):
    """Tampilkan n kata teratas dari setiap klaster."""
    terms = vectorizer.get_feature_names_out()
    print(f"{'Klaster':<10} {'Top Terms'}")
    print('-' * 70)
    for i, center in enumerate(kmeans_model.cluster_centers_):
        top_idx = center.argsort()[-n:][::-1]
        print(f"Klaster {i:<3} {', '.join(terms[j] for j in top_idx)}")


def count_documents_per_cluster(labels):
    """Hitung jumlah dokumen per klaster."""
    unique, counts = np.unique(labels, return_counts=True)
    for cl, cnt in zip(unique, counts):
        print(f"  Klaster {cl}: {cnt} dokumen")


print("Helper functions siap!")

## 5) Clustering dengan TF-IDF

**TF-IDF** (Term Frequency-Inverse Document Frequency) memberi bobot lebih tinggi pada kata yang sering muncul di satu dokumen tetapi jarang di keseluruhan korpus. Ini menghasilkan representasi vektor yang lebih informatif dibanding Bag-of-Words sederhana.

In [ ]:
# Vectorisasi dengan TF-IDF
documents = df['text_final'].dropna().astype(str)
tfidf_vec = TfidfVectorizer(max_features=500)
X_tfidf = tfidf_vec.fit_transform(documents)
print(f"Bentuk matriks TF-IDF: {X_tfidf.shape}")

# Clustering KMeans k=3
kmeans_tfidf, labels_tfidf, sil_tfidf, ch_tfidf, db_tfidf = perform_clustering(X_tfidf, 3)

print("\n=== Hasil Clustering TF-IDF (k=3) ===")
print(f"Silhouette Score     : {sil_tfidf:.4f}")
print(f"Calinski-Harabasz    : {ch_tfidf:.4f}")
print(f"Davies-Bouldin       : {db_tfidf:.4f}")
print("\nJumlah dokumen per klaster:")
count_documents_per_cluster(labels_tfidf)
print("\nKata kunci per klaster:")
get_top_terms(tfidf_vec, kmeans_tfidf)

## 6) Clustering dengan Bag of Words (BoW)

**Bag of Words** menghitung frekuensi kemunculan setiap kata tanpa mempertimbangkan urutan atau konteks. Lebih sederhana dari TF-IDF karena tidak ada pembobotan.

In [ ]:
# Vectorisasi dengan CountVectorizer (BoW)
bow_vec = CountVectorizer(max_features=500)
X_bow = bow_vec.fit_transform(documents)
print(f"Bentuk matriks BoW: {X_bow.shape}")

# Clustering KMeans k=3
kmeans_bow, labels_bow, sil_bow, ch_bow, db_bow = perform_clustering(X_bow, 3)

print("\n=== Hasil Clustering BoW (k=3) ===")
print(f"Silhouette Score     : {sil_bow:.4f}")
print(f"Calinski-Harabasz    : {ch_bow:.4f}")
print(f"Davies-Bouldin       : {db_bow:.4f}")
print("\nJumlah dokumen per klaster:")
count_documents_per_cluster(labels_bow)
print("\nKata kunci per klaster:")
get_top_terms(bow_vec, kmeans_bow)

## 7) Clustering dengan Cosine Similarity

**Cosine Similarity** mengukur sudut antara dua vektor, tidak peduli panjangnya. Cocok untuk dokumen teks karena dua dokumen dianggap mirip jika membicarakan topik yang sama, terlepas dari panjang dokumennya.

In [ ]:
# Hitung Cosine Similarity dari matriks TF-IDF
cos_sim = cosine_similarity(X_tfidf)
print(f"Bentuk matriks Cosine Similarity: {cos_sim.shape}")

# Clustering KMeans pada matriks cosine similarity (dense)
from sklearn.cluster import KMeans as KM
kmeans_cos = KM(n_clusters=3, random_state=42, n_init=10)
labels_cos = kmeans_cos.fit_predict(cos_sim)

sil_cos = silhouette_score(cos_sim, labels_cos)
ch_cos  = calinski_harabasz_score(cos_sim, labels_cos)
db_cos  = davies_bouldin_score(cos_sim, labels_cos)

print("\n=== Hasil Clustering Cosine Similarity (k=3) ===")
print(f"Silhouette Score     : {sil_cos:.4f}")
print(f"Calinski-Harabasz    : {ch_cos:.4f}")
print(f"Davies-Bouldin       : {db_cos:.4f}")
print("\nJumlah dokumen per klaster:")
count_documents_per_cluster(labels_cos)

## 8) Menentukan Jumlah Klaster Optimal

### Elbow Method (WCSS)
Plot nilai WCSS (Within-Cluster Sum of Squares) untuk berbagai nilai k. "Siku" pada grafik menunjukkan k optimal.

### Silhouette Method
Plot rata-rata Silhouette Score untuk berbagai k. Nilai lebih tinggi = klaster lebih kohesif dan terpisah.

In [ ]:
k_range = range(2, 11)
wcss_list = []
sil_list  = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_tfidf)
    wcss_list.append(km.inertia_)
    sil_list.append(silhouette_score(X_tfidf, lbl))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Plot
axes[0].plot(list(k_range), wcss_list, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Jumlah Klaster (k)', fontsize=12)
axes[0].set_ylabel('WCSS', fontsize=12)
axes[0].set_title('Elbow Method — Optimal k untuk TF-IDF', fontsize=13)
axes[0].grid(True, alpha=0.3)

# Silhouette Plot
axes[1].plot(list(k_range), sil_list, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Jumlah Klaster (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Method — Optimal k untuk TF-IDF', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"k optimal (Silhouette tertinggi): k={list(k_range)[sil_list.index(max(sil_list))]}")

## 9) Visualisasi Klaster dengan PCA

**PCA** (Principal Component Analysis) mereduksi dimensi matriks TF-IDF dari ratusan fitur menjadi 2 dimensi sehingga bisa divisualisasikan dalam scatter plot 2D.

In [ ]:
def plot_clusters_pca(X, labels, title, ax):
    """Plot klaster hasil KMeans dalam 2D menggunakan PCA."""
    X_dense = X.toarray() if hasattr(X, 'toarray') else X
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_dense)
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels,
                         cmap='tab10', alpha=0.6, s=20)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(scatter, ax=ax, label='Klaster')


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_clusters_pca(X_tfidf, labels_tfidf, 'Clustering TF-IDF (k=3)', axes[0])
plot_clusters_pca(X_bow,   labels_bow,   'Clustering BoW (k=3)',    axes[1])
plot_clusters_pca(cos_sim, labels_cos,   'Clustering Cosine (k=3)', axes[2])
plt.suptitle('Visualisasi Klaster Honest Review dengan PCA', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 10) Perbandingan Metode

Rangkum dan bandingkan ketiga metode clustering berdasarkan metrik evaluasi.

In [ ]:
comparison = pd.DataFrame({
    'Metode'          : ['TF-IDF', 'Bag of Words', 'Cosine Similarity'],
    'Silhouette Score': [round(sil_tfidf, 4), round(sil_bow, 4), round(sil_cos, 4)],
    'Calinski-Harabasz': [round(ch_tfidf, 2), round(ch_bow, 2), round(ch_cos, 2)],
    'Davies-Bouldin'  : [round(db_tfidf, 4), round(db_bow, 4), round(db_cos, 4)],
})
print("=== Perbandingan Metrik Evaluasi Clustering ===")
print(comparison.to_string(index=False))
print()
best = comparison.loc[comparison['Silhouette Score'].idxmax(), 'Metode']
print(f"✅ Metode terbaik (Silhouette tertinggi): {best}")